[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sbildea/SEED/blob/main/cursuri/06-evaluare-optimizare/extra-06-optuna-studiu-logreg.ipynb)

In [ ]:
# Configurare curs SEED: rulați această celulă prima.
# În Colab descarcă materialele cursului și instalează pachetele lipsă (aprox. un minut).
import os, sys
if "google.colab" in sys.modules:
    if not os.path.exists("/content/SEED"):
        !git clone -q --depth 1 https://github.com/sbildea/SEED.git /content/SEED
        !pip install -q -r /content/SEED/requirements.txt
    %cd /content/SEED/cursuri/06-evaluare-optimizare
sys.path.insert(0, os.path.abspath("../.."))

> **Atenție (exemplu din anii anteriori):** funcția obiectiv de mai jos evaluează modelul pe **setul de test**, deci setul de test influențează alegerea hiperparametrilor (*data leakage*). Varianta corectă, cu validare încrucișată pe antrenare și test folosit o singură dată, este în [seminarul 6](seminar-06-validare-incrucisata-optuna.ipynb).

# Setup studiu - date, variabila tinta

In [ ]:
from sklearn.linear_model import LogisticRegression
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import optuna

# definire nume coloane
col_names = ['pregnant', 'glucose', 'bp', 'skin', 'insulin', 'bmi', 'pedigree', 'age', 'label']

# incarcare set date
pima = pd.read_csv("./date/pima-indians-diabetes.csv", header=0, names=col_names)

#split dataset in features and target variable
feature_cols = ['pregnant', 'insulin', 'bmi', 'age','glucose','bp','pedigree']
X = pima[feature_cols] # Features
y = pima.label # Target variable

# Split dataset into training set and test set
X_train, X_test, y_train, y_test = train_test_split(X, y, 
                                            test_size=0.3, # 70% training and 30% test
                                            random_state=777) # pentru reproductibilitate - in modul debug

# Definire obiectiv de optimizat

In [ ]:
def build_fit_model(params):
    # modelul trebuie construit si antrenat aici explicit
    # DAR parametrii sunt primiti ca argument deci nu trebuie precizati aici
    model = LogisticRegression(
                        **params
                    )
    model.fit(X_train, y_train)
    return model

In [ ]:
def objective(trial):

    # hyperparameter space

    C = trial.suggest_float("C", 0.0625, 512.0)
    l1_ratio = trial.suggest_categorical("l1_ratio", [0.0, 1.0])  # 0 = L2 (Ridge), 1 = L1 (Lasso)

    # build and fit model
    model = build_fit_model(
            params = {
                'C': C,
                'solver': 'liblinear',
                'l1_ratio':l1_ratio
             }
        )
    
    if model is None: 
        print('MODEL could not be loaded')
        return float("inf")

    # Evaluate on the validation set using accuracy metric
    preds = model.predict(X_test)
    acc_score = accuracy_score(y_test, preds)
    return acc_score

In [ ]:
def print_callback(study, trial):

    try:
        print(f"Current trial value: {trial.value}, Current params: {trial.params}")
        print(f"Best study value: {study.best_value}, Best params: {study.best_params}")
    except:
        print('no best params found yet')
        return

In [ ]:
STUDY_NAME='test_logreg_diabetes_full_study'  # Unique identifier of the study.

In [ ]:


try:
    study = optuna.create_study(
        storage="sqlite:///test.sqlite3",
        study_name=STUDY_NAME,
        load_if_exists = False,
        direction="maximize", # maximize accuracy
    )
except:
    study = optuna.load_study(
        storage="sqlite:///test.sqlite3",
        study_name=STUDY_NAME,
        )
study.optimize(objective, 
               #timeout=7200, 
               callbacks=[print_callback], 
               n_trials=1000)

In [ ]:
storage_url="sqlite:///test.sqlite3"
study_name=STUDY_NAME
try:
    study = optuna.load_study(study_name=study_name, storage=storage_url)
    best_params = study.best_trial.params
    print('from best trial')
except Exception as e:
    print(f"Error loading study: {e}")
    # Handle the case where the study doesn't exist, maybe create a new one
    best_params = None
    print('no best trial found')
print(f"Best params: {best_params} \nBest vales: {study.best_value:.2f}")
